# HUMAnN pathway overview of deSWAN transition species (default: window 7.5, age 48)

This notebook describes the stably detectable HUMAnN stratified pathways carried by the deSWAN transition species at the target age.

Inputs and order of execution:

- By default the RDS, breakpoints and window_counts are read from `outputs/deswan-window_7.5/oral_microbiome_*_window_7p5`. `DESWAN_WINDOW` is the width on each side (7.5 gives a total span of about 15 years); the directory keeps `7.5` and file tags use `7p5`.
- The representative main peak of the current results is age 48 (it has not passed the stable main-peak criterion), so `TARGET_AGE=48` by default. Set `TARGET_AGE=49` to follow the original question. The target age can be changed in the configuration cell or through an environment variable.
- The foreground follows the original rule: exact selection of `break_age == TARGET_AGE` from the breakpoint table of the chosen window, or an upstream list that is given explicitly or carries both age and window tags. This set is not the same as all significant species in the window centred on the main peak; legacy lists without a window tag are never used as a fallback.
- The background is recovered from the actual deSWAN RDS and verified against `n_tested` of each window and the header of the original species table.
- A pathway is counted as present when the stratified RPK of the species is above 0 in at least 500 samples. The raw HUMAnN file must be given with `HUMANN_STRATIFIED_FILE`; deSWAN outputs and legacy window summary CSV files cannot replace it.
- The notebook can be started from the project root, the results directory or its subdirectories; set `RESULT_DIR` or `DESWAN_INPUT_DIR` if needed. Age is joined on the unique `participant_id` by default; for a composite key set `AGE_JOIN_KEYS=participant_id,research_stage`.
- After all cells have run, sample-level RPK and label tables are written to `outputs/microbiome_pathway_enrichment` with the default prefix `age48_oral_microbiome_window_7p5`. Run `04-oral_microbiome_plot_pathway.py` afterwards to reproduce the heat map.

Analysis conventions: the dot plot shows only the number of species carrying each pathway, the foreground proportion and the support from positive samples. The one-sided Fisher/BH table remains an exploratory supplement and does not determine the order in the main figure. The age heat map uses sample-level `log1p(RPK)`, a Gaussian kernel with a 2-year bandwidth, a 2-year grid and row Z-scores computed after smoothing; raw age-bin means and kernel effective sample sizes are kept. Age 48 is marked exactly between the grid points at ages 47 and 49.

Figure settings: figure width is 100 mm (including labels and colour bar) and height adapts to wrapped pathway names; no title, subtitle or explanatory text is shown above or below. Axes, value-column headers and the colour-bar label are kept, and the heat map uses a blue–white–red gradient (white at 0).

Interpretive limits: the dot plot reflects the functional composition of candidate species and the heat map a descriptive mean trajectory; neither is equivalent to stable pathway enrichment or a tested age association.


In [ ]:
from pathlib import Path
import gc
import gzip
import json
import os
import re
import textwrap
import warnings

import numpy as np
import pandas as pd
import pyarrow.feather as feather
from scipy.stats import fisher_exact

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory
from IPython.display import display

# ── Analysis configuration ────────────────────────────────────────────────
# Defaults follow outputs/deswan-window_7.5; the representative main peak is age 48.
# DESWAN_WINDOW is the width on each side of the centre age (7.5 ≈ 15-year span).
# Window, target age, input directory and raw HUMAnN file can be set through environment variables.
MICROBIOME_KEY = os.environ.get("MICROBIOME_KEY", "oral_microbiome")
DESWAN_WINDOW = float(os.environ.get("DESWAN_WINDOW", "7.5"))
TARGET_AGE = float(os.environ.get("TARGET_AGE", "48"))
for name, value in [("DESWAN_WINDOW", DESWAN_WINDOW), ("TARGET_AGE", TARGET_AGE)]:
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be finite and positive")
DESWAN_WINDOW_LABEL = f"{DESWAN_WINDOW:g}"
WINDOW_FILE_TAG = "window_" + DESWAN_WINDOW_LABEL.replace(".", "p")
AGE_FILE_TAG = f"{TARGET_AGE:g}".replace(".", "p")
ANALYSIS_PREFIX = f"age{AGE_FILE_TAG}_{MICROBIOME_KEY}_{WINDOW_FILE_TAG}"
DESWAN_PANEL = os.environ.get("DESWAN_PANEL", "panel_a")
MIN_DETECTED_SAMPLES = 500
FDR_ALPHA = 0.05
TOP_N_PATHWAYS = 15
# Figure width includes pathway labels and the colour bar; height adapts to label lines.
FIG_WIDTH_MM = 100
PATHWAY_FONT_PT = 5.5
VALUE_FONT_PT = 5.2
PATHWAY_LABEL_WRAP = 38
MAKE_AGE_HEATMAP = True
AGE_MIN = 40.0
AGE_MAX = 70.0
AGE_BIN_WIDTH = 2.0
AGE_SMOOTHING_BANDWIDTH_YEARS = float(
    os.environ.get("AGE_SMOOTHING_BANDWIDTH_YEARS", "2.0")
)
if not np.isfinite(AGE_SMOOTHING_BANDWIDTH_YEARS) or AGE_SMOOTHING_BANDWIDTH_YEARS <= 0:
    raise ValueError("AGE_SMOOTHING_BANDWIDTH_YEARS must be finite and positive")
if not AGE_MIN <= TARGET_AGE <= AGE_MAX:
    raise ValueError(f"TARGET_AGE must be within [{AGE_MIN:g}, {AGE_MAX:g}]")
# A custom foreground list must come from the chosen window and target age.
TARGET_DESWAN_SPECIES_FILE = None

MICROBIOME_LABELS = {
    "gut_microbiome": "Gut microbiome",
    "oral_microbiome": "Oral microbiome",
}
if MICROBIOME_KEY not in MICROBIOME_LABELS:
    raise ValueError(f"Unsupported MICROBIOME_KEY: {MICROBIOME_KEY}")

# Works from the project root, the results directory or its subdirectories.
if os.environ.get("RESULT_DIR"):
    RESULT_DIR = Path(os.environ["RESULT_DIR"]).expanduser().resolve()
else:
    _result_candidates = []
    for _parent in (Path.cwd(), *Path.cwd().parents):
        _result_candidates.extend([
            _parent,
            _parent / "02-papers" / "Result-02",
        ])
    RESULT_DIR = next((
        p.resolve() for p in _result_candidates
        if (p / "04-oral_microbiome_pathway.ipynb").is_file()
        or (p / "04-oral_microbiome_plot_pathway.py").is_file()
    ), None)
if RESULT_DIR is None or not RESULT_DIR.is_dir():
    raise FileNotFoundError(
        "Could not locate the Result-02 directory; set RESULT_DIR to it."
    )

# Current layout: directories use the decimal 7.5; files use the safe tag window_7p5.
_current_deswan_dir = Path(os.environ.get(
    "DESWAN_INPUT_DIR",
    str(RESULT_DIR / "outputs" / f"deswan-window_{DESWAN_WINDOW_LABEL}"),
)).expanduser()
_current_paths = {
    "rds": _current_deswan_dir / f"{MICROBIOME_KEY}_deswan_breakpoint_{WINDOW_FILE_TAG}.rds",
    "breakpoint": _current_deswan_dir / f"{MICROBIOME_KEY}_breakpoints_{WINDOW_FILE_TAG}.csv",
    "window_count": _current_deswan_dir / f"{MICROBIOME_KEY}_window_counts_{WINDOW_FILE_TAG}.csv",
}
_path_layouts = [("deswan-window directory / window-tagged files", _current_paths)]
# The legacy layout is accepted for the same window only; an explicit input directory disables fallback.
if not os.environ.get("DESWAN_INPUT_DIR"):
    _previous_dir = RESULT_DIR / "outputs" / f"{MICROBIOME_KEY}_pathway"
    _previous_stem = f"{MICROBIOME_KEY}_window_{DESWAN_WINDOW_LABEL}"
    _path_layouts.extend([
        ("previous panel / window-tagged files", {
            "rds": _previous_dir / f"{DESWAN_PANEL}_deswan_breakpoint_{_previous_stem}.rds",
            "breakpoint": _previous_dir / f"{DESWAN_PANEL}_breakpoints_{_previous_stem}.csv",
            "window_count": _previous_dir / f"{DESWAN_PANEL}_window_counts_{_previous_stem}.csv",
        }),
        ("legacy panel_c files in selected window directory", {
            "rds": _current_deswan_dir / f"panel_c_deswan_breakpoint_{MICROBIOME_KEY}.rds",
            "breakpoint": _current_deswan_dir / f"panel_c_breakpoints_{MICROBIOME_KEY}.csv",
            "window_count": _current_deswan_dir / f"panel_c_window_counts_{MICROBIOME_KEY}.csv",
        }),
    ])
DESWAN_LAYOUT, _deswan_paths = next((
    (layout, paths) for layout, paths in _path_layouts
    if all(p.is_file() for p in paths.values())
), _path_layouts[0])
DESWAN_RDS_FILE = _deswan_paths["rds"]
BREAKPOINT_FILE = _deswan_paths["breakpoint"]
WINDOW_COUNT_FILE = _deswan_paths["window_count"]
DESWAN_DIR = DESWAN_RDS_FILE.parent
DESWAN_INPUT_FILE = Path(os.environ.get(
    "DESWAN_INPUT_FILE",
    str(RESULT_DIR.parent / "01-interim_data" / "omics_data"
        / f"{MICROBIOME_KEY}_with_metadata.csv"),
)).expanduser()
AGE_METADATA_FILE = Path(os.environ.get(
    "AGE_METADATA_FILE", str(DESWAN_INPUT_FILE)
)).expanduser()
AGE_JOIN_KEYS = tuple(
    key.strip() for key in os.environ.get("AGE_JOIN_KEYS", "participant_id").split(",")
    if key.strip()
)
if not AGE_JOIN_KEYS or len(set(AGE_JOIN_KEYS)) != len(AGE_JOIN_KEYS):
    raise ValueError("AGE_JOIN_KEYS must contain distinct, non-empty column names")

_humann_defaults = {
    key: Path(
        f"/home/ec2-user/studies/hpp_datasets/{key}/aggregated/"
        "humann_aggregated/pathway_abundance_microbe_level.arrow"
    )
    for key in MICROBIOME_LABELS
}
HUMANN_STRATIFIED_FILE = Path(os.environ.get(
    "HUMANN_STRATIFIED_FILE", str(_humann_defaults[MICROBIOME_KEY])
)).expanduser()
OUTPUT_DIR = Path(os.environ.get(
    "MICROBIOME_PATHWAY_OUTPUT_DIR",
    str(RESULT_DIR / "outputs" / "microbiome_pathway_enrichment"),
)).expanduser()
FIGURE_DIR = Path(os.environ.get(
    "MICROBIOME_PATHWAY_FIGURE_DIR",
    str(RESULT_DIR / "figures" / "microbiome_pathway_enrichment"),
)).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ── Nature style: editable vector text, fonts ≥ 5 pt, restrained colours ────────
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "font.size": 6,
    "savefig.bbox": None,  # fixed canvas on export; tight cropping would change the 100 mm width
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

SIGNAL = "#D97706"
SIGNAL_DARK = "#9A4A00"
NEUTRAL = "#A8A8A8"
NEUTRAL_LIGHT = "#D9D9D9"
TEXT = "#272727"
GRID = "#E8E8E8"
HEATMAP_LOW = "#2166AC"       # blue: negative row Z-score
HEATMAP_NEUTRAL = "#FFFFFF"   # white: zero
HEATMAP_HIGH = "#B2182B"      # red: positive row Z-score
HEATMAP_MISSING = "#D9D9D9"

print(f"Dataset: {MICROBIOME_LABELS[MICROBIOME_KEY]}")
print(f"deSWAN window: {DESWAN_WINDOW} ({DESWAN_LAYOUT})")
print(f"Target age: {TARGET_AGE:g}; output prefix: {ANALYSIS_PREFIX}")
print(f"Result directory: {RESULT_DIR}")
print(f"deSWAN input directory: {DESWAN_DIR}")
print(f"HUMAnN stratified file: {HUMANN_STRATIFIED_FILE}")
if MAKE_AGE_HEATMAP:
    print(f"Age metadata file: {AGE_METADATA_FILE}")
    print(f"Age join keys: {' + '.join(AGE_JOIN_KEYS)}")
    print(
        f"Age smoothing: Gaussian kernel, bandwidth = "
        f"{AGE_SMOOTHING_BANDWIDTH_YEARS:g} years"
    )

In [ ]:
def canonical_species(value):
    """Map full deSWAN taxon names and HUMAnN stratified taxon names to one species key."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None

    tail = str(value).strip().split("|")[-1]
    if "s__" not in tail:
        return None

    species = tail.rsplit("s__", 1)[1]
    species = species.split(".t__", 1)[0]
    species = re.sub(r"[^A-Za-z0-9]+", "_", species).strip("_").casefold()
    return species or None


def extract_taxonomies_from_deswan_rds(rds_file):
    """
    Recover species feature names from the serialized strings of the actual deSWAN RDS.

    For microbiome deSWAN inputs all feature names are k__...|s__.... The names are then
    checked against n_tested in the window-count file and the header of the original deSWAN
    input; the run stops if either check fails, so the background is never silently widened.
    """
    rds_file = Path(rds_file)
    with rds_file.open("rb") as handle:
        magic = handle.read(2)

    opener = gzip.open if magic == b"\x1f\x8b" else open
    with opener(rds_file, "rb") as handle:
        payload = handle.read()

    pattern = rb"k__[A-Za-z0-9_|.\-]+"
    taxa = sorted({match.decode("utf-8") for match in re.findall(pattern, payload)})
    return taxa


def benjamini_hochberg(p_values):
    """Benjamini–Hochberg FDR; q values are returned in input order."""
    p_values = np.asarray(p_values, dtype=float)
    if p_values.ndim != 1:
        raise ValueError("p_values must be one-dimensional")

    n_tests = p_values.size
    if n_tests == 0:
        return np.array([], dtype=float)

    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * n_tests / np.arange(1, n_tests + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0.0, 1.0)

    q_values = np.empty_like(adjusted)
    q_values[order] = adjusted
    return q_values


def split_pathway_label(raw_label):
    """Split a HUMAnN pathway into its ID and readable name."""
    raw_label = str(raw_label).strip()
    if ":" in raw_label:
        pathway_id, pathway_name = raw_label.split(":", 1)
        return pathway_id.strip(), pathway_name.strip()
    return raw_label, ""


def choose_pathway_name(values):
    values = [str(v).strip() for v in values if str(v).strip()]
    if not values:
        return ""
    return pd.Series(values).value_counts().index[0]


def format_q_value(value):
    if not np.isfinite(value):
        return "NA"
    if value < 0.001:
        return f"{value:.1e}"
    return f"{value:.3f}"


def normalize_join_key(series):
    """Normalize join keys across files to stripped, nullable strings."""
    normalized = series.astype("string").str.strip()
    integer_like = normalized.str.fullmatch(
        r"[+-]?\d+\.0+",
        na=False,
    )
    return normalized.mask(
        integer_like,
        normalized.str.replace(r"\.0+$", "", regex=True),
    )


def load_age_metadata(metadata_file, join_keys):
    """Read only the join keys and the existing age; the composite key must be unique."""
    metadata_file = Path(metadata_file)
    if not metadata_file.exists():
        raise FileNotFoundError(
            f"Age metadata file not found: {metadata_file}"
        )

    required_columns = [*join_keys, "age"]
    available_columns = set(
        pd.read_csv(metadata_file, nrows=0).columns.astype(str)
    )
    missing_columns = [
        column
        for column in required_columns
        if column not in available_columns
    ]
    if missing_columns:
        raise KeyError(
            "Age metadata lacks required columns: "
            + ", ".join(missing_columns)
        )

    metadata = pd.read_csv(
        metadata_file,
        usecols=required_columns,
    )
    for key in join_keys:
        metadata[key] = normalize_join_key(metadata[key])
    metadata["age"] = pd.to_numeric(
        metadata["age"],
        errors="coerce",
    )

    missing_key_mask = metadata[list(join_keys)].isna().any(axis=1)
    missing_key_mask |= metadata[list(join_keys)].eq("").any(axis=1)
    if missing_key_mask.any():
        raise RuntimeError(
            f"Age metadata contains {int(missing_key_mask.sum())} rows "
            "with missing join keys."
        )

    duplicated_key_mask = metadata.duplicated(
        list(join_keys),
        keep=False,
    )
    if duplicated_key_mask.any():
        duplicate_examples = metadata.loc[
            duplicated_key_mask,
            required_columns,
        ].head(10)
        raise RuntimeError(
            "Age metadata join keys are not unique; expected one age "
            f"per {' + '.join(join_keys)}. Examples:\n"
            + duplicate_examples.to_string(index=False)
        )
    return metadata


def gaussian_kernel_smooth_by_age(
    ages,
    values,
    age_grid,
    bandwidth_years,
):
    """Gaussian-kernel weighted mean at sample level, with the Kish effective sample size."""
    ages = np.asarray(ages, dtype=float)
    values = np.asarray(values, dtype=float)
    age_grid = np.asarray(age_grid, dtype=float)
    bandwidth_years = float(bandwidth_years)

    if ages.ndim != 1:
        raise ValueError("ages must be one-dimensional")
    if values.ndim != 2 or values.shape[0] != ages.size:
        raise ValueError(
            "values must be a samples-by-pathways matrix aligned to ages"
        )
    if age_grid.ndim != 1 or age_grid.size == 0:
        raise ValueError("age_grid must be a non-empty vector")
    if not np.all(np.diff(age_grid) > 0):
        raise ValueError("age_grid must be strictly increasing")
    if not np.isfinite(bandwidth_years) or bandwidth_years <= 0:
        raise ValueError("bandwidth_years must be finite and positive")
    if not np.isfinite(ages).all() or not np.isfinite(values).all():
        raise ValueError("ages and values must contain only finite values")

    smoothed_columns = []
    effective_sample_sizes = []
    for target_age in age_grid:
        scaled_distance = (ages - target_age) / bandwidth_years
        weights = np.exp(-0.5 * np.square(scaled_distance))
        weight_sum = weights.sum()
        squared_weight_sum = np.square(weights).sum()
        if weight_sum <= 0 or squared_weight_sum <= 0:
            raise RuntimeError(
                f"Gaussian kernel has zero support at age {target_age:g}"
            )
        smoothed_columns.append(
            (weights[:, None] * values).sum(axis=0) / weight_sum
        )
        effective_sample_sizes.append(
            np.square(weight_sum) / squared_weight_sum
        )

    return (
        np.column_stack(smoothed_columns),
        np.asarray(effective_sample_sizes, dtype=float),
    )

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1. Build the strict deSWAN background set and obtain the foreground set at the target age
# ════════════════════════════════════════════════════════════════════════════
required_deswan_files = [
    DESWAN_RDS_FILE,
    BREAKPOINT_FILE,
    WINDOW_COUNT_FILE,
]
missing = [str(path) for path in required_deswan_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required deSWAN files:\n  " + "\n  ".join(missing)
    )

window_counts = pd.read_csv(WINDOW_COUNT_FILE)
n_tested_numeric = pd.to_numeric(
    window_counts["n_tested"],
    errors="coerce",
)
n_tested_missing_count = int(n_tested_numeric.isna().sum())
if n_tested_numeric.empty or not (
    np.isfinite(n_tested_numeric).all()
    and (n_tested_numeric > 0).all()
    and (n_tested_numeric == np.floor(n_tested_numeric)).all()
):
    raise RuntimeError("n_tested must be a non-empty series of positive integers")
expected_n_values = (
    n_tested_numeric.loc[n_tested_numeric.notna()]
    .astype(int)
    .unique()
)
if len(expected_n_values) != 1:
    raise RuntimeError(
        "n_tested is not constant across deSWAN windows; "
        "export the exact tested-feature list from the original run."
    )
expected_n_tested = int(expected_n_values[0])

deswan_taxonomies = extract_taxonomies_from_deswan_rds(DESWAN_RDS_FILE)
if len(deswan_taxonomies) != expected_n_tested:
    raise RuntimeError(
        "Could not recover the exact deSWAN universe: "
        f"RDS yielded {len(deswan_taxonomies)} taxa, "
        f"but window counts report n_tested={expected_n_tested}."
    )

# If the original input table is available, confirm that every species recovered from the RDS was in that deSWAN input.
if DESWAN_INPUT_FILE.exists():
    deswan_input_columns = set(
        pd.read_csv(DESWAN_INPUT_FILE, nrows=0).columns.astype(str)
    )
    rds_not_in_input = sorted(set(deswan_taxonomies) - deswan_input_columns)
    if rds_not_in_input:
        raise RuntimeError(
            "Some RDS taxa are absent from the deSWAN input header; "
            "the cached RDS and input table may not belong to the same run."
        )
else:
    warnings.warn(
        "The deSWAN input CSV is unavailable, so the optional header "
        "cross-check was skipped. The RDS-to-n_tested check still passed."
    )

universe_df = pd.DataFrame({"taxonomy": deswan_taxonomies})
universe_df["species_key"] = universe_df["taxonomy"].map(canonical_species)

if universe_df["species_key"].isna().any():
    raise RuntimeError("At least one deSWAN feature could not be mapped to species level.")
if universe_df["species_key"].duplicated().any():
    duplicated = universe_df.loc[
        universe_df["species_key"].duplicated(keep=False),
        ["taxonomy", "species_key"],
    ]
    raise RuntimeError(
        "Species-key collisions were detected in the deSWAN universe:\n"
        + duplicated.head(10).to_string(index=False)
    )

deswan_universe = set(universe_df["species_key"])

# An explicit foreground list takes precedence; automatic lookup accepts only files tagged with both window and target age.
# Legacy lists without a window tag and crest_lipid_species.txt are never used automatically.
_explicit_foreground = TARGET_DESWAN_SPECIES_FILE or os.environ.get(
    "TARGET_DESWAN_SPECIES_FILE"
)
# The explicitly specified age-49 override remains supported but cannot be used for other ages.
if not _explicit_foreground and TARGET_AGE == 49:
    _explicit_foreground = os.environ.get("AGE49_DESWAN_SPECIES_FILE")
if _explicit_foreground:
    foreground_file = Path(_explicit_foreground).expanduser()
    if not foreground_file.is_file():
        raise FileNotFoundError(f"Explicit foreground file not found: {foreground_file}")
else:
    foreground_candidates = [
        directory / f"age{AGE_FILE_TAG}_deswan_{MICROBIOME_KEY}_{window_tag}_species.txt"
        for directory in (DESWAN_DIR, OUTPUT_DIR, RESULT_DIR / "outputs")
        for window_tag in dict.fromkeys([
            WINDOW_FILE_TAG, f"window_{DESWAN_WINDOW_LABEL}"
        ])
    ]
    foreground_file = next(
        (p for p in foreground_candidates if p.is_file()), None
    )

if foreground_file is not None:
    raw_foreground = [
        line.strip()
        for line in foreground_file.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    foreground_source = f"upstream file: {foreground_file.resolve()}"
else:
    breakpoints = pd.read_csv(BREAKPOINT_FILE)
    missing_columns = {"feature", "break_age"} - set(breakpoints.columns)
    if missing_columns:
        raise KeyError(f"Breakpoint table lacks columns: {sorted(missing_columns)}")
    if "omics" in breakpoints and set(breakpoints["omics"].dropna()) != {MICROBIOME_KEY}:
        raise RuntimeError("Breakpoint table does not match MICROBIOME_KEY")
    break_age = pd.to_numeric(breakpoints["break_age"], errors="coerce")
    foreground_mask = np.isclose(
        break_age,
        TARGET_AGE,
        atol=1e-9,
        rtol=0,
    )
    foreground_feature_series = breakpoints.loc[
        foreground_mask,
        "feature",
    ]
    foreground_missing_feature_count = int(
        foreground_feature_series.isna().sum()
    )
    raw_foreground = (
        foreground_feature_series.loc[
            foreground_feature_series.notna()
        ]
        .astype(str)
        .drop_duplicates()
        .tolist()
    )
    foreground_source = (
        f"fallback: {BREAKPOINT_FILE.name}, break_age = {TARGET_AGE:g}"
    )

unmapped_foreground = [v for v in raw_foreground if canonical_species(v) is None]
if unmapped_foreground:
    raise RuntimeError(f"Foreground contains unrecognized species: {unmapped_foreground[:5]}")
foreground_keys = set(map(canonical_species, raw_foreground))
if not foreground_keys:
    raise RuntimeError(f"No deSWAN foreground taxa have break_age = {TARGET_AGE:g} in {BREAKPOINT_FILE.name}.")

foreground_not_in_universe = sorted(foreground_keys - deswan_universe)
if foreground_not_in_universe:
    raise RuntimeError(
        "The upstream target-age list is not a subset of the selected deSWAN "
        f"universe ({MICROBIOME_KEY}). Example unmatched keys: "
        + ", ".join(foreground_not_in_universe[:10])
    )

foreground_species = foreground_keys
N_UNIVERSE = len(deswan_universe)
N_FOREGROUND = len(foreground_species)

print("deSWAN universe audit passed")
print(f"  deSWAN window:               {DESWAN_WINDOW}")
print(f"  All actually tested species: N = {N_UNIVERSE}")
print(f"  Age-{TARGET_AGE:g} foreground species: n = {N_FOREGROUND}")
print(f"  Foreground source: {foreground_source}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 2. Apply the HUMAnN stratified-pathway presence threshold to all deSWAN species
#    presence = RPK > 0 in at least 500 samples
# ════════════════════════════════════════════════════════════════════════════
if not HUMANN_STRATIFIED_FILE.exists():
    raise FileNotFoundError(
        "HUMAnN stratified abundance file not found: "
        f"{HUMANN_STRATIFIED_FILE}\n"
        "Set the HUMANN_STRATIFIED_FILE environment variable to the correct file."
    )

humann_table = feather.read_table(HUMANN_STRATIFIED_FILE)
df_humann = humann_table.to_pandas()
print(
    f"HUMAnN stratified matrix: "
    f"{df_humann.shape[0]:,} samples × {df_humann.shape[1]:,} columns"
)

feature_records = []
for column in df_humann.columns:
    column_text = str(column)
    if "|" not in column_text:
        continue

    pathway_raw, taxon_raw = column_text.rsplit("|", 1)
    species_key = canonical_species(taxon_raw)
    if species_key is None:
        continue

    pathway_id, pathway_name = split_pathway_label(pathway_raw)
    feature_records.append(
        {
            "column": column,
            "column_label": column_text,
            "pathway_id": pathway_id,
            "pathway_name": pathway_name,
            "species_key": species_key,
        }
    )

feature_info_all = pd.DataFrame(feature_records)
if feature_info_all.empty:
    raise RuntimeError(
        "No species-stratified HUMAnN pathway columns were recognized."
    )

humann_species_before_threshold = set(feature_info_all["species_key"])
feature_info = feature_info_all[
    feature_info_all["species_key"].isin(deswan_universe)
].copy()
if feature_info.empty:
    raise RuntimeError(
        "No HUMAnN species could be mapped to the deSWAN-tested universe."
    )

# Vectorized count of detected samples per column (HUMAnN positive = RPK > 0).
mapped_columns = feature_info["column"].tolist()
try:
    detected_by_column = (df_humann[mapped_columns] > 0).sum(axis=0)
except TypeError:
    numeric_block = df_humann[mapped_columns].apply(
        pd.to_numeric,
        errors="coerce",
    )
    detected_by_column = (numeric_block > 0).sum(axis=0)

feature_info["n_detected_samples"] = (
    feature_info["column"].map(detected_by_column).astype(int)
)

# Normal HUMAnN output has one column per pathway–species pair; for duplicated mappings,
# recount from the union of samples with RPK > 0 in any duplicate column.
pair_sizes = feature_info.groupby(
    ["pathway_id", "species_key"],
    sort=False,
).size()
duplicate_pairs = pair_sizes[pair_sizes > 1]

edge_counts = (
    feature_info.groupby(
        ["pathway_id", "species_key"],
        as_index=False,
        sort=False,
    )
    .agg(
        n_detected_samples=("n_detected_samples", "max"),
        n_source_columns=("column", "nunique"),
    )
)

if not duplicate_pairs.empty:
    for pathway_id, species_key in duplicate_pairs.index:
        duplicate_columns = feature_info.loc[
            (feature_info["pathway_id"] == pathway_id)
            & (feature_info["species_key"] == species_key),
            "column",
        ].tolist()
        try:
            detected_union = (
                df_humann[duplicate_columns] > 0
            ).any(axis=1).sum()
        except TypeError:
            duplicate_block = df_humann[duplicate_columns].apply(
                pd.to_numeric,
                errors="coerce",
            )
            detected_union = (duplicate_block > 0).any(axis=1).sum()

        mask = (
            (edge_counts["pathway_id"] == pathway_id)
            & (edge_counts["species_key"] == species_key)
        )
        edge_counts.loc[mask, "n_detected_samples"] = int(detected_union)

pathway_names = (
    feature_info.groupby("pathway_id")["pathway_name"]
    .agg(choose_pathway_name)
    .rename("pathway_name")
    .reset_index()
)
edge_counts = edge_counts.merge(pathway_names, on="pathway_id", how="left")

species_pathway_edges = edge_counts[
    edge_counts["n_detected_samples"] >= MIN_DETECTED_SAMPLES
].copy()

if species_pathway_edges.empty:
    raise RuntimeError(
        "No pathway–species pair passed the RPK > 0 in at least "
        f"{MIN_DETECTED_SAMPLES} samples criterion."
    )

# Keep a compact matrix for the descriptive age heat map, then release the full HUMAnN matrix.
age_heatmap_sample_metadata = None
age_heatmap_pathway_rpk = None
AGE_HEATMAP_PATHWAY_IDS = []
if MAKE_AGE_HEATMAP:
    foreground_threshold_edges = species_pathway_edges[
        species_pathway_edges["species_key"].isin(foreground_species)
    ].copy()
    if foreground_threshold_edges.empty:
        raise RuntimeError(
            "No threshold-qualified foreground edge is available "
            "for the descriptive age heatmap."
        )

    heatmap_selection = (
        foreground_threshold_edges.groupby(
            ["pathway_id", "pathway_name"],
            as_index=False,
            dropna=False,
        )
        .agg(
            foreground_taxa_with_pathway=("species_key", "nunique"),
            rpk_positive_samples_median=(
                "n_detected_samples",
                "median",
            ),
        )
        .sort_values(
            [
                "foreground_taxa_with_pathway",
                "rpk_positive_samples_median",
                "pathway_id",
            ],
            ascending=[False, False, True],
            kind="mergesort",
        )
    )
    AGE_HEATMAP_PATHWAY_IDS = heatmap_selection[
        "pathway_id"
    ].head(TOP_N_PATHWAYS).tolist()

    heatmap_pairs = foreground_threshold_edges[
        foreground_threshold_edges["pathway_id"].isin(
            AGE_HEATMAP_PATHWAY_IDS
        )
    ][["pathway_id", "species_key"]].drop_duplicates()
    heatmap_feature_info = feature_info.merge(
        heatmap_pairs,
        on=["pathway_id", "species_key"],
        how="inner",
    )
    if heatmap_feature_info.empty:
        raise RuntimeError(
            "Could not recover HUMAnN columns for the age heatmap."
        )

    heatmap_columns = heatmap_feature_info["column"].tolist()
    heatmap_numeric = df_humann[heatmap_columns].apply(
        pd.to_numeric,
        errors="coerce",
    ).fillna(0.0)
    age_heatmap_pathway_rpk = pd.DataFrame(
        index=pd.RangeIndex(len(df_humann), name="_humann_row")
    )

    # For duplicated pathway–species columns, take the per-sample maximum to avoid double counting;
    # then sum over candidate species passing the threshold to obtain their pathway RPK.
    for pathway_id in AGE_HEATMAP_PATHWAY_IDS:
        pathway_total = np.zeros(len(df_humann), dtype=float)
        pathway_species = heatmap_pairs.loc[
            heatmap_pairs["pathway_id"] == pathway_id,
            "species_key",
        ]
        for species_key in pathway_species:
            pair_columns = heatmap_feature_info.loc[
                (heatmap_feature_info["pathway_id"] == pathway_id)
                & (
                    heatmap_feature_info["species_key"]
                    == species_key
                ),
                "column",
            ].tolist()
            pair_values = heatmap_numeric[pair_columns].max(
                axis=1
            )
            pathway_total += pair_values.to_numpy(dtype=float)
        age_heatmap_pathway_rpk[pathway_id] = pathway_total

    metadata_candidates = list(AGE_JOIN_KEYS)
    index_metadata = df_humann.index.to_frame(index=False)
    age_heatmap_sample_metadata = pd.DataFrame(
        {"_humann_row": np.arange(len(df_humann), dtype=int)}
    )
    for metadata_column in metadata_candidates:
        if metadata_column in index_metadata.columns:
            age_heatmap_sample_metadata[metadata_column] = (
                index_metadata[metadata_column].to_numpy()
            )
        elif metadata_column in df_humann.columns:
            age_heatmap_sample_metadata[metadata_column] = (
                df_humann[metadata_column].to_numpy()
            )

    missing_humann_join_keys = [
        key
        for key in AGE_JOIN_KEYS
        if key not in age_heatmap_sample_metadata.columns
    ]
    if missing_humann_join_keys:
        raise RuntimeError(
            "HUMAnN row metadata lacks required age-join keys: "
            + ", ".join(missing_humann_join_keys)
        )

    del heatmap_numeric

mapped_universe_n = len(
    deswan_universe & humann_species_before_threshold
)
mapped_foreground_n = len(
    foreground_species & humann_species_before_threshold
)

print("HUMAnN mapping and thresholding complete")
print(
    f"  deSWAN species represented in HUMAnN stratification: "
    f"{mapped_universe_n}/{N_UNIVERSE}"
)
print(
    f"  Age-{TARGET_AGE:g} species represented in HUMAnN stratification: "
    f"{mapped_foreground_n}/{N_FOREGROUND}"
)
print(
    f"  Thresholded pathway–species occurrences: "
    f"{len(species_pathway_edges):,}"
)
print(
    f"  Unique pathways after thresholding: "
    f"{species_pathway_edges['pathway_id'].nunique():,}"
)
if MAKE_AGE_HEATMAP:
    print(
        f"  Compact age-heatmap matrix: "
        f"{age_heatmap_pathway_rpk.shape[0]:,} samples × "
        f"{age_heatmap_pathway_rpk.shape[1]:,} pathways"
    )

# Release the full HUMAnN matrix to reduce memory use during plotting and export.
for large_object_name in [
    "df_humann",
    "humann_table",
    "numeric_block",
    "duplicate_block",
]:
    globals().pop(large_object_name, None)
gc.collect()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 3. Exploratory supplement: one-sided Fisher test (not used for the main figure or confirmatory conclusions)
# ════════════════════════════════════════════════════════════════════════════
present_by_pathway = (
    species_pathway_edges.groupby("pathway_id")["species_key"]
    .agg(lambda values: set(values))
)

pathway_name_map = (
    species_pathway_edges.drop_duplicates("pathway_id")
    .set_index("pathway_id")["pathway_name"]
    .to_dict()
)

enrichment_rows = []
for pathway_id, present_species in present_by_pathway.items():
    present_species = set(present_species) & deswan_universe

    # a/b: target-age foreground; c/d: remaining deSWAN species in the background; the four cells are mutually exclusive.
    a = len(present_species & foreground_species)
    b = N_FOREGROUND - a
    K = len(present_species)
    c = K - a
    d = (N_UNIVERSE - N_FOREGROUND) - c

    if min(a, b, c, d) < 0:
        raise RuntimeError(
            f"Invalid contingency table for pathway {pathway_id}: "
            f"{[a, b, c, d]}"
        )

    odds_ratio, p_value = fisher_exact(
        [[a, b], [c, d]],
        alternative="greater",
    )

    foreground_proportion = a / N_FOREGROUND
    universe_proportion = K / N_UNIVERSE
    fold_enrichment = (
        foreground_proportion / universe_proportion
        if universe_proportion > 0
        else np.nan
    )

    enrichment_rows.append(
        {
            "pathway_id": pathway_id,
            "pathway_name": pathway_name_map.get(pathway_id, ""),
            "foreground_with_pathway": a,
            "foreground_without_pathway": b,
            "nonforeground_with_pathway": c,
            "nonforeground_without_pathway": d,
            "foreground_n": N_FOREGROUND,
            "deswan_universe_with_pathway": K,
            "deswan_universe_n": N_UNIVERSE,
            "foreground_proportion": foreground_proportion,
            "deswan_universe_proportion": universe_proportion,
            "proportion_difference": (
                foreground_proportion - universe_proportion
            ),
            "fold_enrichment": fold_enrichment,
            "odds_ratio": odds_ratio,
            "p_fisher_greater": p_value,
        }
    )

enrichment_results = pd.DataFrame(enrichment_rows)
enrichment_results["q_bh"] = benjamini_hochberg(
    enrichment_results["p_fisher_greater"].values
)
enrichment_results["significant"] = (
    (enrichment_results["q_bh"] < FDR_ALPHA)
    & (enrichment_results["fold_enrichment"] > 1)
)

enrichment_results = enrichment_results.sort_values(
    ["q_bh", "p_fisher_greater", "fold_enrichment"],
    ascending=[True, True, False],
    kind="mergesort",
).reset_index(drop=True)

N_TESTED_PATHWAYS = len(enrichment_results)
N_SIGNIFICANT_PATHWAYS = int(enrichment_results["significant"].sum())

print("Exploratory pathway over-representation testing complete")
print(f"  Pathways tested: {N_TESTED_PATHWAYS:,}")
print(
    f"  BH q < {FDR_ALPHA:g} and fold enrichment > 1: "
    f"{N_SIGNIFICANT_PATHWAYS:,}"
)
print("  Interpret as hypothesis-generating only: the foreground is small.")
display(enrichment_results.head(10))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 4. Save full results, background set and mapping audit
# ════════════════════════════════════════════════════════════════════════════
thresholded_pathway_counts = (
    species_pathway_edges.groupby("species_key")["pathway_id"]
    .nunique()
    .rename("n_thresholded_pathways")
)

universe_audit = universe_df.copy()
universe_audit["is_target_age_foreground"] = universe_audit[
    "species_key"
].isin(foreground_species)
universe_audit["represented_in_humann_stratification"] = universe_audit[
    "species_key"
].isin(humann_species_before_threshold)
universe_audit = universe_audit.merge(
    thresholded_pathway_counts,
    left_on="species_key",
    right_index=True,
    how="left",
)
universe_audit["n_thresholded_pathways"] = (
    universe_audit["n_thresholded_pathways"].fillna(0).astype(int)
)

foreground_audit = universe_audit[
    universe_audit["is_target_age_foreground"]
].copy()

all_results_file = OUTPUT_DIR / (
    f"{ANALYSIS_PREFIX}_"
    "pathway_enrichment_exploratory_all.csv"
)
edge_file = OUTPUT_DIR / (
    f"{MICROBIOME_KEY}_{WINDOW_FILE_TAG}_"
    "thresholded_species_pathway_edges.csv"
)
universe_file = OUTPUT_DIR / (
    f"{ANALYSIS_PREFIX}_"
    "deswan_tested_species_universe.csv"
)
foreground_file_out = OUTPUT_DIR / (
    f"{ANALYSIS_PREFIX}_"
    "deswan_species.csv"
)

enrichment_results.to_csv(all_results_file, index=False)
species_pathway_edges.sort_values(
    ["pathway_id", "species_key"]
).to_csv(edge_file, index=False)
universe_audit.to_csv(universe_file, index=False)
foreground_audit.to_csv(foreground_file_out, index=False)

analysis_metadata = {
    "microbiome_key": MICROBIOME_KEY,
    "deswan_window": DESWAN_WINDOW,
    "deswan_layout": DESWAN_LAYOUT,
    "window_file_tag": WINDOW_FILE_TAG,
    "deswan_rds_file": str(DESWAN_RDS_FILE.resolve()),
    "breakpoint_file": str(BREAKPOINT_FILE.resolve()),
    "window_count_file": str(WINDOW_COUNT_FILE.resolve()),
    "humann_stratified_file": str(HUMANN_STRATIFIED_FILE.resolve()),
    "age_metadata_file": str(AGE_METADATA_FILE.resolve()),
    "age_join_keys": list(AGE_JOIN_KEYS),
    "foreground_selection_rule": (
        "explicit/window-and-age-tagged species list, otherwise exact break_age == target_age; "
        "this is not the set of all significant taxa in the peak-centred window"
    ),
    "target_age": TARGET_AGE,
    "foreground_source": foreground_source,
    "pathway_occurrence_rule": (
        f"species-stratified HUMAnN RPK > 0 in at least "
        f"{MIN_DETECTED_SAMPLES} samples"
    ),
    "background_definition": (
        "all taxa recovered from the actual deSWAN RDS and validated "
        "against the constant n_tested reported across deSWAN windows"
    ),
    "deswan_universe_n": N_UNIVERSE,
    "foreground_n": N_FOREGROUND,
    "humann_mapped_universe_n": mapped_universe_n,
    "humann_mapped_foreground_n": mapped_foreground_n,
    "tested_pathways_n": N_TESTED_PATHWAYS,
    "significant_pathways_n": N_SIGNIFICANT_PATHWAYS,
    "main_figure_role": (
        "descriptive functional overview; no inferential significance "
        "encoded"
    ),
    "age_heatmap_role": (
        "descriptive mean trajectory only; no age-association test"
    ),
    "age_heatmap_aggregation": (
        "sum eligible species-stratified pathway RPK per sample; "
        "log1p transform; Gaussian-kernel weighted mean on a fixed "
        "age grid; row z-score"
    ),
    "age_heatmap_range": [AGE_MIN, AGE_MAX],
    "age_heatmap_bin_width_years": AGE_BIN_WIDTH,
    "age_heatmap_smoothing_method": (
        "sample-level Gaussian-kernel weighted mean"
    ),
    "age_heatmap_smoothing_bandwidth_years": (
        AGE_SMOOTHING_BANDWIDTH_YEARS
    ),
    "exploratory_test": (
        "one-sided Fisher exact test, alternative='greater'"
    ),
    "exploratory_multiple_testing": "Benjamini-Hochberg FDR",
    "inference_boundary": (
        "hypothesis-generating only because the target-age foreground "
        "contains few taxa and therefore has limited statistical power"
    ),
    "fdr_alpha": FDR_ALPHA,
}
metadata_file = OUTPUT_DIR / (
    f"{ANALYSIS_PREFIX}_"
    "pathway_analysis_metadata.json"
)
metadata_file.write_text(
    json.dumps(analysis_metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

mapping_rate = mapped_universe_n / N_UNIVERSE
if mapping_rate < 0.80:
    warnings.warn(
        f"Only {mapping_rate:.1%} of the deSWAN universe is represented in "
        "the HUMAnN stratified labels. Inspect the universe audit for "
        "taxonomy-version mismatches before interpreting pathway patterns."
    )

print("Traceable outputs saved")
print(f"  {all_results_file}")
print(f"  {edge_file}")
print(f"  {universe_file}")
print(f"  {foreground_file_out}")
print(f"  {metadata_file}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5. Nature-style main-figure panel: descriptive functional overview of pathways passing the threshold
#    Encodes only the number of species and of detected samples, not P, q or fold enrichment.
# ════════════════════════════════════════════════════════════════════════════
foreground_edges = species_pathway_edges[
    species_pathway_edges["species_key"].isin(foreground_species)
].copy()
if foreground_edges.empty:
    raise RuntimeError(
        f"No threshold-qualified pathway occurs in the age-{TARGET_AGE:g} foreground taxa."
    )

foreground_edges["pathway_name"] = (
    foreground_edges["pathway_name"].fillna("").astype(str)
)
pathway_overview = (
    foreground_edges.groupby(
        ["pathway_id", "pathway_name"],
        as_index=False,
        dropna=False,
    )
    .agg(
        foreground_taxa_with_pathway=("species_key", "nunique"),
        rpk_positive_samples_median=("n_detected_samples", "median"),
        rpk_positive_samples_min=("n_detected_samples", "min"),
        rpk_positive_samples_max=("n_detected_samples", "max"),
    )
)

universe_pathway_counts = (
    species_pathway_edges.groupby("pathway_id")["species_key"]
    .nunique()
    .rename("deswan_universe_taxa_with_pathway")
    .reset_index()
)
pathway_overview = pathway_overview.merge(
    universe_pathway_counts,
    on="pathway_id",
    how="left",
)
pathway_overview["foreground_taxa_n"] = N_FOREGROUND
pathway_overview["deswan_universe_n"] = N_UNIVERSE
pathway_overview["foreground_taxa_proportion"] = (
    pathway_overview["foreground_taxa_with_pathway"] / N_FOREGROUND
)
pathway_overview["eligibility_rule"] = (
    f"species-stratified RPK > 0 in at least "
    f"{MIN_DETECTED_SAMPLES} samples"
)

# Ordering is based on descriptive evidence only, not on Fisher P or BH q values.
pathway_overview = pathway_overview.sort_values(
    [
        "foreground_taxa_with_pathway",
        "rpk_positive_samples_median",
        "pathway_id",
    ],
    ascending=[False, False, True],
    kind="mergesort",
).reset_index(drop=True)
pathway_overview["descriptive_rank"] = np.arange(
    1,
    len(pathway_overview) + 1,
)

N_FOREGROUND_PATHWAYS = len(pathway_overview)
plot_data = pathway_overview.head(TOP_N_PATHWAYS).copy()


def make_display_label(row):
    pathway_id = str(row["pathway_id"])
    pathway_name = str(row["pathway_name"]).strip()
    if pathway_name.casefold() == "nan":
        pathway_name = ""
    combined = (
        f"{pathway_id}  {pathway_name}"
        if pathway_name
        else pathway_id
    )
    return textwrap.fill(combined, width=PATHWAY_LABEL_WRAP)


def format_sample_count(value):
    value = float(value)
    if value.is_integer():
        return f"{int(value):,}"
    return f"{value:,.1f}"


plot_data["display_label"] = plot_data.apply(
    make_display_label,
    axis=1,
)

n_rows = len(plot_data)
max_label_lines = max(label.count("\n") + 1 for label in plot_data["display_label"])
pathway_row_height_mm = max(
    4.2,
    (max_label_lines * PATHWAY_FONT_PT * 1.2 + 3.0) * 25.4 / 72,
)
fig_width_in = FIG_WIDTH_MM / 25.4
# Reserve one line at the top for value-column headers and only the axis space at the bottom.
fig_height_mm = max(60.0, (n_rows + 1) * pathway_row_height_mm + 12.0)
fig_height_in = fig_height_mm / 25.4
fig, ax = plt.subplots(
    figsize=(fig_width_in, fig_height_in),
    facecolor="white",
)

y = np.arange(n_rows)
taxa_counts = plot_data["foreground_taxa_with_pathway"].to_numpy(dtype=float)

# Segments and filled points show the exact number of foreground species; they carry no significance.
ax.hlines(
    y,
    xmin=0,
    xmax=taxa_counts,
    color=NEUTRAL_LIGHT,
    linewidth=0.8,
    zorder=1,
)
ax.scatter(
    taxa_counts,
    y,
    s=16,
    facecolor=SIGNAL,
    edgecolor=SIGNAL_DARK,
    linewidth=0.55,
    zorder=3,
)

ax.set_yticks(y)
ax.set_yticklabels(
    plot_data["display_label"],
    fontsize=PATHWAY_FONT_PT,
    color=TEXT,
)
ax.tick_params(axis="y", length=0, pad=3)
ax.tick_params(
    axis="x",
    labelsize=5.5,
    colors=TEXT,
    length=2,
    width=0.7,
)
ax.set_xlabel(
    f"Age-{TARGET_AGE:g} taxa, n",
    fontsize=6.0,
    color=TEXT,
    labelpad=3,
)
ax.xaxis.set_major_locator(
    mpl.ticker.MaxNLocator(
        integer=True,
        nbins=min(6, max(2, int(np.nanmax(taxa_counts)) + 1)),
    )
)
ax.xaxis.grid(True, color=GRID, linewidth=0.6)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_color("#767676")

max_taxa_count = int(np.nanmax(taxa_counts))
ax.set_xlim(0, max(1.0, max_taxa_count + 0.35))
ax.set_ylim(n_rows - 0.5, -1.5)

# The value columns on the right show the small denominators and threshold evidence, not inferential statistics.
annotation_transform = blended_transform_factory(
    ax.transAxes,
    ax.transData,
)
column_x = {
    "taxa": 1.21,
    "samples": 2.10,
}
header_y = -0.66
header_texts = []
for x_position, header in [
    (column_x["taxa"], f"Age {TARGET_AGE:g}\na/n"),
    (
        column_x["samples"],
        "RPK-positive samples\nmedian (range)",
    ),
]:
    header_texts.append(
        ax.text(
            x_position,
            header_y,
            header,
            transform=annotation_transform,
            ha="center",
            va="bottom",
            fontsize=VALUE_FONT_PT,
            fontweight="bold",
            color=TEXT,
            clip_on=False,
        )
    )

annotation_texts = []
for yi, row in zip(y, plot_data.itertuples(index=False)):
    sample_median = format_sample_count(
        row.rpk_positive_samples_median
    )
    sample_min = format_sample_count(row.rpk_positive_samples_min)
    sample_max = format_sample_count(row.rpk_positive_samples_max)
    sample_label = f"{sample_median} ({sample_min}–{sample_max})"
    values = [
        (
            column_x["taxa"],
            f"{row.foreground_taxa_with_pathway}/{row.foreground_taxa_n}",
        ),
        (column_x["samples"], sample_label),
    ]
    for x_position, label in values:
        annotation_texts.append(
            ax.text(
                x_position,
                yi,
                label,
                transform=annotation_transform,
                ha="center",
                va="center",
                fontsize=VALUE_FONT_PT,
                color=TEXT,
                clip_on=False,
            )
        )

# No title, subtitle or caption; value-column headers are kept to explain the data.
fig.subplots_adjust(
    left=0.43,
    right=0.64,
    top=1.0 - 2.0 / fig_height_mm,
    bottom=10.0 / fig_height_mm,
)

# Audit font sizes and text collisions after rendering (single plot area, so no panel-alignment audit).
fig.canvas.draw()
renderer = fig.canvas.get_renderer()
visible_text = [
    text_object
    for text_object in fig.findobj(mpl.text.Text)
    if text_object.get_visible() and text_object.get_text().strip()
]
visible_font_sizes = [
    text_object.get_fontsize() for text_object in visible_text
]
minimum_rendered_source_font_pt = min(visible_font_sizes)
if minimum_rendered_source_font_pt < 5:
    raise RuntimeError(
        "Figure contains source text below the 5 pt floor: "
        f"{minimum_rendered_source_font_pt:.2f} pt"
    )


def find_text_collisions(text_objects, active_renderer):
    boxes = [
        text_object.get_window_extent(active_renderer).expanded(1.01, 1.03)
        for text_object in text_objects
    ]
    collisions = []
    for left_index in range(len(boxes)):
        for right_index in range(left_index + 1, len(boxes)):
            if boxes[left_index].overlaps(boxes[right_index]):
                collisions.append(
                    (
                        text_objects[left_index].get_text(),
                        text_objects[right_index].get_text(),
                    )
                )
    return collisions


collision_groups = {
    "pathway_labels": find_text_collisions(
        [label for label in ax.get_yticklabels() if label.get_visible()],
        renderer,
    ),
    "right_value_columns": find_text_collisions(
        annotation_texts,
        renderer,
    ),
    "right_headers_and_values": find_text_collisions(
        header_texts + annotation_texts,
        renderer,
    ),
}
collision_count = sum(
    len(collisions) for collisions in collision_groups.values()
)
if collision_count:
    raise RuntimeError(
        "Text collision audit failed: "
        + json.dumps(collision_groups, ensure_ascii=False)
    )

figure_base = FIGURE_DIR / (
    f"{ANALYSIS_PREFIX}_"
    "eligible_pathway_overview"
)
figure_paths = {
    "svg": figure_base.with_name(f"{figure_base.name}.svg"),
    "pdf": figure_base.with_name(f"{figure_base.name}.pdf"),
    "png": figure_base.with_name(f"{figure_base.name}.png"),
}
fig.savefig(
    figure_paths["svg"],
    bbox_inches=None,
    pad_inches=0,
    facecolor="white",
)
fig.savefig(
    figure_paths["pdf"],
    bbox_inches=None,
    pad_inches=0,
    facecolor="white",
)
fig.savefig(
    figure_paths["png"],
    dpi=300,
    bbox_inches=None,
    pad_inches=0,
    facecolor="white",
)

svg_text = figure_paths["svg"].read_text(encoding="utf-8")
if "<text" not in svg_text:
    raise RuntimeError("SVG text was converted to paths; editable text is required.")

plot_data_file = OUTPUT_DIR / (
    f"{ANALYSIS_PREFIX}_"
    "eligible_pathway_overview_plot_data.csv"
)
all_overview_file = OUTPUT_DIR / (
    f"{ANALYSIS_PREFIX}_"
    "eligible_pathway_overview_all.csv"
)
plot_data.to_csv(plot_data_file, index=False)
pathway_overview.to_csv(all_overview_file, index=False)

figure_qa = {
    "archetype": "ranked descriptive dot plot with direct-value columns",
    "evidence_role": "descriptive main-figure subpanel",
    "core_conclusion": (
        "The displayed pathways satisfy the prespecified HUMAnN occurrence "
        f"criterion in at least one age-{TARGET_AGE:g} deSWAN taxon; exact a/n and sample "
        "coverage are shown without inferential significance."
    ),
    "selection_rule": (
        f"top {TOP_N_PATHWAYS} pathways by foreground taxa count, then median "
        f"RPK-positive sample count; all eligible pathways are in source data"
    ),
    "figure_width_mm": FIG_WIDTH_MM,
    "figure_height_mm": fig_height_mm,
    "pathway_label_wrap_characters": PATHWAY_LABEL_WRAP,
    "title_subtitle_footnote": "omitted",
    "alignment_gate": "not applicable: one scientific plotting axis",
    "minimum_source_font_pt": minimum_rendered_source_font_pt,
    "collision_audit": {
        "status": "passed",
        "n_detected": collision_count,
        "groups": collision_groups,
    },
    "editable_svg_text": True,
    "vector_exports": ["svg", "pdf"],
    "raster_exports": ["png_300_dpi"],
    "statistics_in_figure": "none; descriptive counts only",
    "exploratory_statistics": (
        "one-sided Fisher exact test with BH correction is exported "
        "separately and is not used for ranking or visual encoding"
    ),
    "deswan_window": DESWAN_WINDOW,
    "target_age": TARGET_AGE,
    "foreground_n": N_FOREGROUND,
    "pathway_occurrence_threshold_samples": MIN_DETECTED_SAMPLES,
}
qa_file = OUTPUT_DIR / (
    f"{ANALYSIS_PREFIX}_"
    "eligible_pathway_overview_figure_qa.json"
)
qa_file.write_text(
    json.dumps(figure_qa, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(fig)
plt.close(fig)

print("Descriptive publication subpanel and source data saved")
for format_name, path in figure_paths.items():
    print(f"  {format_name.upper()}: {path}")
print(f"  Plot data: {plot_data_file}")
print(f"  All eligible pathways: {all_overview_file}")
print(f"  Figure QA: {qa_file}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6. Descriptive age heat map: mean pathway RPK contributed by candidate species across age
#    No correlation, regression or significance filtering.
# ════════════════════════════════════════════════════════════════════════════
if not MAKE_AGE_HEATMAP:
    print("Descriptive age heatmap is disabled by configuration.")
else:
    if age_heatmap_pathway_rpk is None:
        raise RuntimeError(
            "The compact pathway RPK matrix was not prepared in the HUMAnN cell."
        )
    age_lookup = load_age_metadata(
        AGE_METADATA_FILE,
        AGE_JOIN_KEYS,
    )
    humann_metadata = age_heatmap_sample_metadata.copy()
    for key in AGE_JOIN_KEYS:
        humann_metadata[key] = normalize_join_key(
            humann_metadata[key]
        )
    missing_humann_key_mask = humann_metadata[
        list(AGE_JOIN_KEYS)
    ].isna().any(axis=1)
    missing_humann_key_mask |= humann_metadata[
        list(AGE_JOIN_KEYS)
    ].eq("").any(axis=1)
    if missing_humann_key_mask.any():
        raise RuntimeError(
            f"HUMAnN metadata contains "
            f"{int(missing_humann_key_mask.sum())} rows with missing "
            f"join keys: {', '.join(AGE_JOIN_KEYS)}."
        )

    age_metadata = humann_metadata.merge(
        age_lookup,
        on=list(AGE_JOIN_KEYS),
        how="left",
        validate="many_to_one",
        indicator="_age_join_status",
    )
    age_metadata["age"] = pd.to_numeric(
        age_metadata["age"],
        errors="coerce",
    )
    age_metadata["_humann_row"] = pd.to_numeric(
        age_metadata["_humann_row"],
        errors="coerce",
    )
    age_join_audit = age_metadata[[
        "_humann_row",
        *AGE_JOIN_KEYS,
        "age",
        "_age_join_status",
    ]].copy()
    age_join_audit["has_valid_age"] = age_join_audit[
        "age"
    ].notna()
    age_join_audit_file = OUTPUT_DIR / (
        f"{ANALYSIS_PREFIX}_"
        "eligible_pathway_age_join_audit.csv"
    )
    age_join_audit.to_csv(age_join_audit_file, index=False)

    n_humann_metadata_rows = len(age_metadata)
    n_age_key_matches = int(
        age_metadata["_age_join_status"].eq("both").sum()
    )
    n_valid_joined_ages = int(
        (
            age_metadata["_age_join_status"].eq("both")
            & age_metadata["age"].notna()
        ).sum()
    )
    age_join_rate = (
        n_age_key_matches / n_humann_metadata_rows
        if n_humann_metadata_rows
        else np.nan
    )
    if n_age_key_matches == 0:
        raise RuntimeError(
            "No HUMAnN sample matched the age metadata on "
            f"{' + '.join(AGE_JOIN_KEYS)}."
        )
    if n_age_key_matches < n_humann_metadata_rows:
        warnings.warn(
            f"{n_humann_metadata_rows - n_age_key_matches:,} of "
            f"{n_humann_metadata_rows:,} HUMAnN rows did not match age "
            "metadata and will be excluded; see the join audit CSV."
        )

    age_metadata = age_metadata.loc[
        age_metadata["_age_join_status"].eq("both")
        & age_metadata["age"].notna()
        & age_metadata["_humann_row"].notna()
    ].copy()
    age_metadata = age_metadata[
        age_metadata["age"].between(
            AGE_MIN,
            AGE_MAX,
            inclusive="both",
        )
    ].copy()
    age_metadata["_humann_row"] = (
        age_metadata["_humann_row"].astype(int)
    )
    if age_metadata["_humann_row"].duplicated().any():
        raise RuntimeError(
            "Age join duplicated HUMAnN row identifiers despite the "
            "many-to-one validation."
        )
    if age_metadata.empty:
        raise RuntimeError(
            f"No HUMAnN samples have valid ages between "
            f"{AGE_MIN:g} and {AGE_MAX:g} years."
        )

    pathway_order = plot_data["pathway_id"].tolist()
    if pathway_order != AGE_HEATMAP_PATHWAY_IDS:
        raise RuntimeError(
            "The pathway order selected for the dot plot and age heatmap "
            "does not match; rerun all cells from the HUMAnN threshold step."
        )
    missing_pathways = [
        pathway_id
        for pathway_id in pathway_order
        if pathway_id not in age_heatmap_pathway_rpk.columns
    ]
    if missing_pathways:
        raise RuntimeError(
            "Compact age-heatmap matrix is missing pathways: "
            + ", ".join(missing_pathways[:5])
        )

    humann_rows = age_metadata["_humann_row"].to_numpy(dtype=int)
    pathway_rpk = (
        age_heatmap_pathway_rpk.loc[humann_rows, pathway_order]
        .reset_index(drop=True)
        .clip(lower=0)
    )
    age_analysis = pathway_rpk.copy()
    age_analysis["age"] = age_metadata["age"].to_numpy(dtype=float)
    age_analysis["participant_id"] = (
        age_metadata["participant_id"].astype(str).to_numpy()
    )

    age_bin_edges = np.arange(
        AGE_MIN,
        AGE_MAX + AGE_BIN_WIDTH * 0.5,
        AGE_BIN_WIDTH,
    )
    if not np.isclose(age_bin_edges[-1], AGE_MAX):
        age_bin_edges = np.append(age_bin_edges, AGE_MAX)
    n_age_bins = len(age_bin_edges) - 1
    age_analysis["_age_bin"] = pd.cut(
        age_analysis["age"],
        bins=age_bin_edges,
        labels=False,
        include_lowest=True,
        right=True,
    )
    age_analysis = age_analysis.dropna(subset=["_age_bin"]).copy()
    age_analysis["_age_bin"] = age_analysis["_age_bin"].astype(int)
    if age_analysis["_age_bin"].nunique() < 3:
        raise RuntimeError(
            "Fewer than three non-empty age bins are available; "
            "the descriptive trajectory is not interpretable."
        )

    bin_index = pd.Index(
        np.arange(n_age_bins),
        name="_age_bin",
    )
    age_bin_sample_n = (
        age_analysis.groupby("_age_bin")
        .size()
        .reindex(bin_index, fill_value=0)
        .astype(int)
    )
    age_bin_participant_n = (
        age_analysis.groupby("_age_bin")["participant_id"]
        .nunique()
        .reindex(bin_index, fill_value=0)
        .astype(int)
    )
    mean_rpk_by_age = (
        age_analysis.groupby("_age_bin")[pathway_order]
        .mean()
        .reindex(bin_index)
        .T
    )
    sample_log1p_pathway_rpk = np.log1p(
        age_analysis[pathway_order]
    )
    log1p_pathway_rpk = sample_log1p_pathway_rpk.copy()
    log1p_pathway_rpk["_age_bin"] = age_analysis[
        "_age_bin"
    ].to_numpy()
    mean_log1p_rpk_by_age = (
        log1p_pathway_rpk.groupby("_age_bin")[pathway_order]
        .mean()
        .reindex(bin_index)
        .T
    )

    age_bin_midpoint = (
        age_bin_edges[:-1] + age_bin_edges[1:]
    ) / 2
    (
        smoothed_log1p_values,
        kernel_effective_sample_n,
    ) = gaussian_kernel_smooth_by_age(
        ages=age_analysis["age"].to_numpy(dtype=float),
        values=sample_log1p_pathway_rpk.to_numpy(dtype=float),
        age_grid=age_bin_midpoint,
        bandwidth_years=AGE_SMOOTHING_BANDWIDTH_YEARS,
    )
    smoothed_log1p_rpk_by_age = pd.DataFrame(
        smoothed_log1p_values,
        index=pathway_order,
        columns=bin_index,
    )

    row_mean = smoothed_log1p_rpk_by_age.mean(
        axis=1,
        skipna=True,
    )
    row_sd = smoothed_log1p_rpk_by_age.std(
        axis=1,
        skipna=True,
        ddof=0,
    ).replace(0, np.nan)
    age_heatmap_z = smoothed_log1p_rpk_by_age.sub(
        row_mean,
        axis=0,
    ).div(
        row_sd,
        axis=0,
    )
    constant_rows = age_heatmap_z.isna().all(axis=1)
    if constant_rows.any():
        warnings.warn(
            f"{int(constant_rows.sum())} pathway rows have no smoothed "
            "variation and will be displayed at the neutral colour."
        )
        age_heatmap_z.loc[constant_rows] = 0.0
    pathway_label_map = (
        plot_data.fillna({"pathway_name": ""})
        .set_index("pathway_id")["pathway_name"].to_dict()
    )
    from html import unescape

    # Use original names so earlier wrapping cannot insert spaces into IDs.
    pathway_label_map = {
        pathway_id: unescape(f"{pathway_id} {pathway_name}").strip()
        for pathway_id, pathway_name in pathway_label_map.items()
    }
    heatmap_label_font_pt = 5.2
    heatmap_line_spacing = 1.2
    heatmap_label_width_mm = FIG_WIDTH_MM * 0.44
    heatmap_outer_margin_mm = 0.4
    heatmap_colorbar_height_mm = 2.2
    heatmap_colorbar_gap_mm = 1.2

    age_fig_width_in = FIG_WIDTH_MM / 25.4
    age_fig, age_ax = plt.subplots(
        figsize=(age_fig_width_in, 80.0 / 25.4),
        facecolor="white",
    )
    age_fig.canvas.draw()
    age_renderer = age_fig.canvas.get_renderer()
    heatmap_label_font = mpl.font_manager.FontProperties(
        family=mpl.rcParams["font.family"],
        size=heatmap_label_font_pt,
    )

    def wrap_heatmap_label(label, renderer, font_properties, max_width_px):
        """Wrap complete labels using rendered width, including long identifiers."""
        def text_width(value):
            return renderer.get_text_width_height_descent(
                value, font_properties, ismath=False,
            )[0]

        lines = []
        current = ""
        for word in label.split():
            candidate = f"{current} {word}" if current else word
            if text_width(candidate) <= max_width_px:
                current = candidate
                continue
            if current:
                lines.append(current)
            current = ""
            while text_width(word) > max_width_px:
                split_at = 1
                while (split_at < len(word)
                       and text_width(word[:split_at + 1]) <= max_width_px):
                    split_at += 1
                lines.append(word[:split_at])
                word = word[split_at:]
            current = word
        if current:
            lines.append(current)
        return "\n".join(lines)

    pathway_label_map = {
        pathway_id: wrap_heatmap_label(
            label, age_renderer, heatmap_label_font,
            heatmap_label_width_mm / 25.4 * age_fig.dpi,
        )
        for pathway_id, label in pathway_label_map.items()
    }

    heatmap_cmap = mpl.colors.LinearSegmentedColormap.from_list(
        "blue_white_red",
        [HEATMAP_LOW, HEATMAP_NEUTRAL, HEATMAP_HIGH],
        N=256,
    )
    heatmap_cmap.set_bad(HEATMAP_MISSING)
    heatmap_values = np.ma.masked_invalid(
        age_heatmap_z.loc[pathway_order].to_numpy(dtype=float)
    )
    age_image = age_ax.imshow(
        heatmap_values,
        aspect="auto",
        interpolation="nearest",
        cmap=heatmap_cmap,
        norm=mpl.colors.TwoSlopeNorm(
            vmin=-2,
            vcenter=0,
            vmax=2,
        ),
    )

    age_ax.set_yticks(np.arange(len(pathway_order)))
    age_ax.set_yticklabels(
        [pathway_label_map[pathway_id] for pathway_id in pathway_order],
        fontsize=heatmap_label_font_pt,
        linespacing=heatmap_line_spacing,
        color=TEXT,
    )
    age_tick_positions = np.unique(
        np.linspace(
            0,
            n_age_bins - 1,
            min(5, n_age_bins),
            dtype=int,
        )
    )
    age_ax.set_xticks(age_tick_positions)
    age_ax.set_xticklabels(
        [
            f"{age_bin_midpoint[position]:g}"
            for position in age_tick_positions
        ],
        fontsize=5.5,
        color=TEXT,
    )
    age_ax.set_xlabel(
        "Age (years)",
        fontsize=6.0,
        color=TEXT,
        labelpad=3,
    )
    age_ax.tick_params(axis="y", length=0, pad=3)
    age_ax.tick_params(
        axis="x",
        length=2,
        width=0.7,
        colors=TEXT,
    )

    age_ax.set_xticks(
        np.arange(-0.5, n_age_bins, 1),
        minor=True,
    )
    age_ax.set_yticks(
        np.arange(-0.5, len(pathway_order), 1),
        minor=True,
    )
    age_ax.grid(
        which="minor",
        color="white",
        linewidth=0.3,
    )
    age_ax.tick_params(which="minor", length=0)
    for spine in age_ax.spines.values():
        spine.set_visible(False)

    # Mark the target age exactly; age 48 lies between the grid points at 47 and 49, and the smoothing grid is unchanged.
    target_grid_position = (TARGET_AGE - AGE_MIN) / AGE_BIN_WIDTH - 0.5
    if np.isclose(target_grid_position, round(target_grid_position)):
        target_outline = mpl.patches.Rectangle(
            (target_grid_position - 0.5, -0.5),
            1,
            len(pathway_order),
            fill=False,
            edgecolor=TEXT,
            linewidth=0.7,
            zorder=5,
        )
        age_ax.add_patch(target_outline)
    else:
        age_ax.vlines(
            target_grid_position, -0.5, len(pathway_order) - 0.5,
            color=TEXT, linewidth=0.7, zorder=5,
        )
    age_ax.set_ylim(len(pathway_order) - 0.5, -0.5)

    # A separate horizontal colorbar does not reserve an empty right column.
    age_colorbar_ax = age_fig.add_axes([0.48, 0.92, 0.50, 0.028])
    age_colorbar = age_fig.colorbar(
        age_image,
        cax=age_colorbar_ax,
        orientation="horizontal",
        ticks=[-2, 0, 2],
    )
    age_colorbar.ax.xaxis.set_ticks_position("top")
    age_colorbar.set_label(
        "Row Z-score", fontsize=5.5, color=TEXT,
        horizontalalignment="right", verticalalignment="center",
    )
    age_colorbar.ax.xaxis.set_label_position("top")
    age_colorbar.ax.xaxis.set_label_coords(-0.035, 0.5)
    age_colorbar.ax.tick_params(
        labelsize=5.2, length=1.5, pad=1.5, width=0.6, colors=TEXT,
    )
    # Keep the endpoint labels within the colorbar's horizontal extent.
    age_colorbar.ax.get_xticklabels()[0].set_ha("left")
    age_colorbar.ax.get_xticklabels()[-1].set_ha("right")
    age_colorbar.outline.set_linewidth(0.6)

    # Measure real text extents and fit the axes inside the fixed 100 mm canvas.
    # Repeating after resizing also handles font/backend metric differences.
    for _layout_pass in range(3):
        age_fig.canvas.draw()
        age_renderer = age_fig.canvas.get_renderer()
        px_to_mm = 25.4 / age_fig.dpi
        pathway_boxes = [
            label.get_window_extent(age_renderer)
            for label in age_ax.get_yticklabels() if label.get_visible()
        ]
        heatmap_row_height_mm = max(
            3.6,
            max(box.height for box in pathway_boxes) * px_to_mm + 0.7,
        )
        heatmap_plot_height_mm = len(pathway_order) * heatmap_row_height_mm
        plot_box = age_ax.get_window_extent(age_renderer)
        plot_tight_box = age_ax.get_tightbbox(age_renderer)
        colorbar_box = age_colorbar.ax.get_window_extent(age_renderer)
        colorbar_tight_box = age_colorbar.ax.get_tightbbox(age_renderer)
        left_mm = heatmap_outer_margin_mm + max(
            0.0,
            plot_box.x0 - plot_tight_box.x0,
            colorbar_box.x0 - colorbar_tight_box.x0,
        ) * px_to_mm
        right_mm = heatmap_outer_margin_mm + max(
            0.0,
            plot_tight_box.x1 - plot_box.x1,
            colorbar_tight_box.x1 - colorbar_box.x1,
        ) * px_to_mm
        bottom_mm = heatmap_outer_margin_mm + max(
            0.0, plot_box.y0 - plot_tight_box.y0,
        ) * px_to_mm
        colorbar_bottom_extra_mm = max(
            0.0, colorbar_box.y0 - colorbar_tight_box.y0,
        ) * px_to_mm
        colorbar_top_extra_mm = max(
            0.0, colorbar_tight_box.y1 - colorbar_box.y1,
        ) * px_to_mm
        colorbar_bottom_mm = (
            bottom_mm + heatmap_plot_height_mm
            + heatmap_colorbar_gap_mm + colorbar_bottom_extra_mm
        )
        age_fig_height_mm = (
            colorbar_bottom_mm + heatmap_colorbar_height_mm
            + colorbar_top_extra_mm + heatmap_outer_margin_mm
        )
        plot_width_mm = FIG_WIDTH_MM - left_mm - right_mm
        if plot_width_mm <= 0:
            raise RuntimeError("Heatmap labels leave no plotting width.")
        age_fig.set_size_inches(FIG_WIDTH_MM / 25.4, age_fig_height_mm / 25.4)
        age_ax.set_position([
            left_mm / FIG_WIDTH_MM,
            bottom_mm / age_fig_height_mm,
            plot_width_mm / FIG_WIDTH_MM,
            heatmap_plot_height_mm / age_fig_height_mm,
        ])
        age_colorbar.ax.set_position([
            left_mm / FIG_WIDTH_MM,
            colorbar_bottom_mm / age_fig_height_mm,
            plot_width_mm / FIG_WIDTH_MM,
            heatmap_colorbar_height_mm / age_fig_height_mm,
        ])

    age_fig.canvas.draw()
    age_renderer = age_fig.canvas.get_renderer()
    age_visible_text = [
        text_object
        for text_object in age_fig.findobj(mpl.text.Text)
        if text_object.get_visible() and text_object.get_text().strip()
    ]
    age_clipped_text = []
    for text_object in age_visible_text:
        text_box = text_object.get_window_extent(age_renderer)
        if (text_box.x0 < age_fig.bbox.x0 or text_box.y0 < age_fig.bbox.y0
                or text_box.x1 > age_fig.bbox.x1 or text_box.y1 > age_fig.bbox.y1):
            age_clipped_text.append(text_object.get_text())
    if age_clipped_text:
        raise RuntimeError(
            "Heatmap text extends beyond the canvas: "
            + json.dumps(age_clipped_text, ensure_ascii=False)
        )

    age_minimum_font_pt = min(
        text_object.get_fontsize()
        for text_object in age_visible_text
    )
    if age_minimum_font_pt < 5:
        raise RuntimeError(
            "Age heatmap contains text below the 5 pt floor: "
            f"{age_minimum_font_pt:.2f} pt"
        )

    age_collision_groups = {
        "all_visible_labels": find_text_collisions(
            age_visible_text, age_renderer,
        ),
        "pathway_labels": find_text_collisions(
            [
                label
                for label in age_ax.get_yticklabels()
                if label.get_visible()
            ],
            age_renderer,
        ),
        "age_ticks": find_text_collisions(
            [
                label
                for label in age_ax.get_xticklabels()
                if label.get_visible()
            ],
            age_renderer,
        ),
    }
    age_collision_count = sum(
        len(collisions)
        for collisions in age_collision_groups.values()
    )
    if age_collision_count:
        raise RuntimeError(
            "Age heatmap text collision audit failed: "
            + json.dumps(
                age_collision_groups,
                ensure_ascii=False,
            )
        )

    age_figure_base = FIGURE_DIR / (
        f"{ANALYSIS_PREFIX}_"
        "eligible_pathway_age_heatmap"
    )
    age_figure_paths = {
        "svg": age_figure_base.with_name(f"{age_figure_base.name}.svg"),
        "pdf": age_figure_base.with_name(f"{age_figure_base.name}.pdf"),
        "png": age_figure_base.with_name(f"{age_figure_base.name}.png"),
    }
    age_fig.savefig(
        age_figure_paths["svg"],
        bbox_inches=None,
        pad_inches=0,
        facecolor="white",
    )
    age_fig.savefig(
        age_figure_paths["pdf"],
        bbox_inches=None,
        pad_inches=0,
        facecolor="white",
    )
    age_fig.savefig(
        age_figure_paths["png"],
        dpi=300,
        bbox_inches=None,
        pad_inches=0,
        facecolor="white",
    )

    age_svg_text = age_figure_paths["svg"].read_text(encoding="utf-8")
    if "<text" not in age_svg_text:
        raise RuntimeError(
            "Age-heatmap SVG text was converted to paths; "
            "editable text is required."
        )

    pathway_names_for_export = (
        plot_data.set_index("pathway_id")["pathway_name"].to_dict()
    )
    supporting_taxa_for_export = (
        plot_data.set_index("pathway_id")[
            "foreground_taxa_with_pathway"
        ].to_dict()
    )
    age_heatmap_records = []
    for pathway_id in pathway_order:
        for age_bin_index in range(n_age_bins):
            age_heatmap_records.append(
                {
                    "pathway_id": pathway_id,
                    "pathway_name": pathway_names_for_export[pathway_id],
                    "age_bin_start": age_bin_edges[age_bin_index],
                    "age_bin_end": age_bin_edges[age_bin_index + 1],
                    "age_bin_interval": (
                        ("[" if age_bin_index == 0 else "(")
                        + f"{age_bin_edges[age_bin_index]:g}, "
                        + f"{age_bin_edges[age_bin_index + 1]:g}]"
                    ),
                    "age_bin_midpoint": age_bin_midpoint[age_bin_index],
                    "smoothing_method": (
                        "sample-level Gaussian-kernel weighted mean"
                    ),
                    "smoothing_bandwidth_years": (
                        AGE_SMOOTHING_BANDWIDTH_YEARS
                    ),
                    "kernel_effective_sample_n": (
                        kernel_effective_sample_n[age_bin_index]
                    ),
                    "n_samples": int(
                        age_bin_sample_n.loc[age_bin_index]
                    ),
                    "n_participants": int(
                        age_bin_participant_n.loc[age_bin_index]
                    ),
                    "n_supporting_foreground_taxa": int(
                        supporting_taxa_for_export[pathway_id]
                    ),
                    "mean_summed_rpk": mean_rpk_by_age.loc[
                        pathway_id,
                        age_bin_index,
                    ],
                    "mean_log1p_summed_rpk": mean_log1p_rpk_by_age.loc[
                        pathway_id,
                        age_bin_index,
                    ],
                    "smoothed_mean_log1p_summed_rpk": (
                        smoothed_log1p_rpk_by_age.loc[
                            pathway_id,
                            age_bin_index,
                        ]
                    ),
                    "row_z_score": age_heatmap_z.loc[
                        pathway_id,
                        age_bin_index,
                    ],
                }
            )
    age_heatmap_source = pd.DataFrame(age_heatmap_records)
    age_heatmap_source_file = OUTPUT_DIR / (
        f"{ANALYSIS_PREFIX}_"
        "eligible_pathway_age_heatmap_source_data.csv"
    )
    age_heatmap_source.to_csv(
        age_heatmap_source_file,
        index=False,
    )

    age_bin_audit = pd.DataFrame(
        {
            "age_bin_start": age_bin_edges[:-1],
            "age_bin_end": age_bin_edges[1:],
            "age_bin_interval": [
                ("[" if index == 0 else "(")
                + f"{age_bin_edges[index]:g}, "
                + f"{age_bin_edges[index + 1]:g}]"
                for index in range(n_age_bins)
            ],
            "age_bin_midpoint": age_bin_midpoint,
            "n_samples": age_bin_sample_n.to_numpy(),
            "n_participants": age_bin_participant_n.to_numpy(),
            "kernel_effective_sample_n": (
                kernel_effective_sample_n
            ),
        }
    )
    age_bin_audit_file = OUTPUT_DIR / (
        f"{ANALYSIS_PREFIX}_"
        "eligible_pathway_age_bin_audit.csv"
    )
    age_bin_audit.to_csv(age_bin_audit_file, index=False)

    age_figure_qa = {
        "archetype": "single-panel descriptive age heatmap",
        "evidence_role": "descriptive secondary main-figure subpanel",
        "core_conclusion": (
            "Threshold-qualified candidate-taxon pathway contributions "
            "show heterogeneous smoothed average profiles across age."
        ),
        "aggregation": (
            "sum eligible species-stratified RPK per pathway and sample; "
            "log1p transform; sample-level Gaussian-kernel weighted "
            "mean on a fixed age grid; row z-score"
        ),
        "smoothing_method": (
            "Nadaraya-Watson Gaussian-kernel weighted mean"
        ),
        "smoothing_bandwidth_years": (
            AGE_SMOOTHING_BANDWIDTH_YEARS
        ),
        "smoothing_support_metric": (
            "Kish effective sample size exported at every age-grid point"
        ),
        "normalization": (
            "row z-score calculated after smoothing; absolute raw and "
            "log1p bin means retained in Source Data"
        ),
        "colour_scale": {
            "type": "symmetric diverging, fixed at [-2, 2]",
            "low": HEATMAP_LOW,
            "midpoint": HEATMAP_NEUTRAL,
            "high": HEATMAP_HIGH,
            "missing": HEATMAP_MISSING,
            "accessibility": (
                "Okabe-Ito blue/vermillion endpoints; no rainbow or "
                "red-green-only encoding"
            ),
        },
        "age_range": [AGE_MIN, AGE_MAX],
        "deswan_window": DESWAN_WINDOW,
        "target_age": TARGET_AGE,
        "target_age_grid_position": target_grid_position,
        "age_bin_width_years": AGE_BIN_WIDTH,
        "age_bin_closure": (
            "right-closed; the first bin also includes the lower bound"
        ),
        "age_metadata_file": str(AGE_METADATA_FILE),
        "age_join_keys": list(AGE_JOIN_KEYS),
        "n_humann_metadata_rows": n_humann_metadata_rows,
        "n_age_key_matches": n_age_key_matches,
        "n_valid_joined_ages": n_valid_joined_ages,
        "age_key_match_rate": age_join_rate,
        "n_samples_with_age": int(len(age_analysis)),
        "alignment_gate": (
            "not applicable: one scientific heatmap axis; colour bar excluded"
        ),
        "figure_width_mm": FIG_WIDTH_MM,
        "figure_height_mm": age_fig_height_mm,
        "pathway_label_max_width_mm": heatmap_label_width_mm,
        "label_wrapping": "measured font width",
        "colorbar_position": "top_horizontal",
        "outer_margin_mm": heatmap_outer_margin_mm,
        "clipped_text_count": len(age_clipped_text),
        "pathway_row_height_mm": heatmap_row_height_mm,
        "title_subtitle_footnote": "omitted",
        "minimum_source_font_pt": age_minimum_font_pt,
        "collision_audit": {
            "status": "passed",
            "n_detected": age_collision_count,
            "groups": age_collision_groups,
        },
        "editable_svg_text": True,
        "vector_exports": ["svg", "pdf"],
        "raster_exports": ["png_300_dpi"],
        "statistics_in_figure": (
            "none; descriptive kernel-smoothed means and row z-scores only"
        ),
        "review_risks": [
            "visual trajectory depends on the prespecified bandwidth",
            "row z-scores suppress between-pathway absolute differences",
            "edge estimates may have lower effective support",
        ],
    }
    age_qa_file = OUTPUT_DIR / (
        f"{ANALYSIS_PREFIX}_"
        "eligible_pathway_age_heatmap_figure_qa.json"
    )
    age_qa_file.write_text(
        json.dumps(
            age_figure_qa,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    display(age_fig)
    plt.close(age_fig)

    print("Descriptive pathway age heatmap and source data saved")
    for format_name, path in age_figure_paths.items():
        print(f"  {format_name.upper()}: {path}")
    print(f"  Source data: {age_heatmap_source_file}")
    print(f"  Age-join audit: {age_join_audit_file}")
    print(
        f"  Age-key matches: {n_age_key_matches:,}/"
        f"{n_humann_metadata_rows:,} ({age_join_rate:.1%})"
    )
    print(
        f"  Gaussian smoothing bandwidth: "
        f"{AGE_SMOOTHING_BANDWIDTH_YEARS:g} years"
    )
    print(
        f"  Kernel effective sample size: "
        f"{kernel_effective_sample_n.min():.1f}–"
        f"{kernel_effective_sample_n.max():.1f}"
    )
    print(f"  Age-bin audit: {age_bin_audit_file}")
    print(f"  Figure QA: {age_qa_file}")

In [ ]:
if MAKE_AGE_HEATMAP:
    # Save sample-level pathway RPK data
    sample_level_rpk = age_analysis[
        [
            "participant_id",
            "age",
            "_age_bin",
            *pathway_order,
        ]
    ].copy()

    sample_level_rpk = sample_level_rpk.rename(
        columns={"_age_bin": "age_bin_index"}
    )

    sample_level_rpk_file = OUTPUT_DIR / (
        f"{ANALYSIS_PREFIX}_"
        "eligible_pathway_age_heatmap_sample_level_rpk.csv"
    )

    sample_level_rpk.to_csv(
        sample_level_rpk_file,
        index=False,
    )

    print(f"Sample-level pathway RPK saved: {sample_level_rpk_file}")

## Interpretation and audit order

1. Start with the deSWAN species background audit table and confirm that N equals n_tested of every window of the original deSWAN run.
2. Check the coverage of HUMAnN taxon-name mapping; unmapped species stay in the deSWAN denominator, but differences in taxonomy version should be investigated.
3. The main figure shows only pathways that pass the threshold in at least one target-age species, ordered by the number of target-age species carrying the pathway and then by the median number of RPK-positive samples.
4. Orange points and segments show the number of species; `a/n` on the right states the small foreground denominator, and the sample column gives the median and range across the species–pathway pairs supporting the pathway. P values, BH q values and fold enrichment are not encoded.
5. The Fisher/BH table is an exploratory reference only. Because few species are present at the target age, non-significance cannot be read as no functional difference, and a nominally high fold cannot be read as stable enrichment.
6. Age is read from the existing `age` field of `*_with_metadata.csv` and joined to HUMAnN samples by a strict many-to-one merge on the configured `AGE_JOIN_KEYS` (default `participant_id`); the full matching status is saved as an audit table.
7. For the age heat map, the stratified pathway RPK of candidate species passing the threshold is summed per sample and `log1p`-transformed; weighted means are then computed on a 2-year age grid with a Gaussian kernel of 2-year bandwidth, and row Z-scores per pathway are computed after smoothing.
8. Source Data keep the unsmoothed absolute age-bin means, the smoothed means, row Z-scores and the kernel effective sample size at each grid point, so that smoothing does not hide the support in the raw data.
9. The heat map uses a diverging blue–white–red palette with a fixed midpoint at 0; colour shows the level relative to the mean of that pathway and cannot be used to compare absolute abundance between pathways.
10. The heat map remains a descriptive mean trajectory without correlation, regression or significance testing; the shape depends on the preset bandwidth, and the effective support at boundary ages should be judged with the audit table.
11. Because the foreground contains few species, the conclusions support functional composition and candidate age trajectories only, not stable enrichment or a statistically tested age association.

Main outputs: the complete thresholded pathway–species edge table, Source Data of the dot plot and age heat map, the age-join audit, the age-bin sample-size audit, the deSWAN background audit table, the foreground species table, the exploratory Fisher/BH table, and SVG, PDF and 300 dpi PNG figures. TIFF is not produced under the current conventions. All file names contain the safe deSWAN window tag (for example `window_7p5`); outputs specific to the target age also contain the age tag.